# Sampling timing parameters with vela-jax and Discovery

vela-jax evaluates Vela.jl's delay chain as a **JAX function of the timing
parameters**. That one property is what lets a sampler treat the timing model
like any other part of the likelihood: `r(theta)` is `jit`-able and
`jacfwd`-able, so NUTS can differentiate through it instead of calling out to
tempo2 once per step.

This notebook takes a par/tim pair to a corner plot of the orbital parameters:

```
TimingPulsar.from_files(par, tim)   # a pulsar, read once
  -> TimingSpec(engines="vela_jax") # what to sample
  -> discovery_signals()            # a Discovery likelihood
  -> NUTS -> corner
```

`vela_jax.TimingPulsar` implements nltiming's `TimingPulsar` protocol.
`timing_package` chooses who *opens the files*. PINT is the default and is
the right reader for NANOGrav and for Vela.jl's simulations. InPTA, EPTA and
IPTA files want tempo2: that is their native timing package. The delay
physics is Vela's either way.

The walkthrough loads both. First a PINT-native ELL1 simulation
(`J1802-2124.sim`). Then **InPTA narrowband J0613-0200** — 399 real TOAs from
the uGMRT, `BINARY ELL1` — read by tempo2. The sampling cells use the InPTA
pulsar. Both files live in Vela.jl's example fixtures.

Needs `discovery`, `nltiming`, `numpyro` and `corner`. The InPTA half also
needs the `tempo2` extra (libstempo + tempo2).

In [ ]:
import os
import sys

os.environ.setdefault("JAX_ENABLE_X64", "1")

from pathlib import Path

import corner
import discovery as ds
import jax
import matplotlib.pyplot as plt
import nltiming.sampling as nlts
import numpy as np
from loguru import logger
from nltiming import TimingSpec
from numpyro.infer import init_to_value

from vela_jax import TimingPulsar

logger.remove()
logger.add(sys.stderr, level="WARNING")

nlts.numpyro.ensure_x64()

# Discovery's decentering chart (`MarginalTransport`) lives on the metamath
# kernel path; select it before building any model.
ds.config(kernels="metamath")

## The fixtures

Vela.jl's example fixtures, shared with vela-jax's own test suite. Point
`VELA_JAX_FIXTURES` at a copy of `pyvela/examples` if it is not the sibling
checkout.

In [ ]:
FIXTURES = Path(
    os.environ.get("VELA_JAX_FIXTURES")
    or Path.cwd().resolve().parents[1] / "Vela.jl" / "pyvela" / "examples"
)
assert FIXTURES.is_dir(), f"fixtures not found under {FIXTURES}"
print(FIXTURES)

## A PINT pulsar

One read, one object. `timing_package` says who opens the files. This ELL1
simulation was written for PINT, so PINT reads it. Nothing below depends on
that choice except the freeze: the delay physics is Vela's either way.

In [ ]:
pint_par = FIXTURES / "J1802-2124.sim.par"
pint_tim = FIXTURES / "J1802-2124.sim.tim"
assert pint_par.is_file() and pint_tim.is_file()

pint_pulsar = TimingPulsar.from_files(pint_par, pint_tim, timing_package="pint")

print(f"{pint_pulsar.name}: {len(pint_pulsar.toas)} TOAs, "
      f"timing_package={pint_pulsar.engine.timing_package}, "
      f"{pint_pulsar.residuals.std() * 1e6:.2f} us residual rms")

## An InPTA pulsar, read by tempo2

InPTA is a tempo2 PTA. The same `from_files` call, with
`timing_package="tempo2"` and `binary_conventions="tempo2"` so the ELL1
truncation matches the package that fitted the par. The rest of the notebook
samples this pulsar.

In [ ]:
PAR = FIXTURES / "J0613-0200.InPTA.NB.par"
TIM = FIXTURES / "J0613-0200.InPTA.NB.tim"
assert PAR.is_file() and TIM.is_file()

pulsar = TimingPulsar.from_files(
    PAR, TIM, timing_package="tempo2", binary_conventions="tempo2"
)

print(f"{pulsar.name}: {len(pulsar.toas)} TOAs, "
      f"timing_package={pulsar.engine.timing_package}, "
      f"conventions={pulsar.engine.binary_conventions}, "
      f"{pulsar.residuals.std() * 1e6:.2f} us residual rms")

Loading this par raises one warning worth reading rather than silencing:
the InPTA par sets `CORRECT_TROPOSPHERE Y`, and vela-jax does not model a
tropospheric delay. It says so, and it says what the consequence is — the
*absolute* residuals carry it as an offset of order 10 ns, while residual
*differences*, which is what a sampler moves through, are unaffected. An
engine that refuses to guess, and tells you the size of what it left out, is
the behaviour to want here.

It is an `nltiming` `TimingPulsar` in its own right, not a duck that happens to fit:

In [ ]:
from nltiming.protocols import TimingPulsar as TimingPulsarProtocol

print("satisfies the protocol:", isinstance(pulsar, TimingPulsarProtocol))
print("can serve vela_jax:    ", pulsar.can_use_engines("vela_jax"))

## The differentiable residual

This is the part that makes the rest possible. Two invariants are worth seeing
directly rather than taking on trust:

* `residual_delta(0) == 0` exactly — the engine is *at* its expansion point, so
  a zero parameter step is a zero residual change, with no round-off floor.
* `M == -J` exactly — the design matrix handed to the likelihood **is** the
  engine's own Jacobian, not a second matrix computed some other way. A residual
  and a linearisation that disagree is a whole class of bug that cannot arise
  here.

In [ ]:
engine = pulsar.timing_engine("vela_jax")
zero = np.zeros(len(engine.fitpars))

print("fitpars:", engine.fitpars)
print("residual_delta(0), max |.|:", np.max(np.abs(engine.residual_delta(zero))))

J = engine.residual_jacobian()
M = engine.design_matrix()
print("jacobian shape:", J.shape)
print("max |M + J|:   ", np.max(np.abs(M + J)))

## What to sample

`TimingSpec` chooses the inference plan; the engine computes the delay. The
default plan samples the axes the timing model is nonlinear in and
analytically marginalizes the rest.

Pass `engines` as a plain string. A partial mapping such as
`{"pint": "vela_jax"}` leaves the other timing package at its default, and a
vela-jax pulsar refuses a mixed request rather than quietly serving half of it.

nltiming warns that the marginalized axes here are only *locally* linear. That
is honest bookkeeping, not an error: their integration is a local
approximation around the expansion point. Declaring axes identically linear,
and certifying the geometry before a long run, is its own topic.

In [ ]:
spec = TimingSpec(engines="vela_jax", name="timing")
timing = spec.for_pulsar(pulsar)

print("sampled:     ", timing.sampled)
print("marginalized:", timing.marginalized)

## A Discovery likelihood

`discovery_signals()` supplies the timing pieces; the rest is ordinary
Discovery. The noise model is deliberately minimal — a fixed `EFAC = 1`, no red
noise — so the figure is about the timing parameters and the run is quick.
**That also means the posterior should not be expected to reproduce the
published par exactly**: those values came from a fit with a real noise model.

`decentered_model` samples the timing block in a whitened frame — the sampler
moves in a standard-normal `xi` and the transport carries it to physical
coordinates, which removes the funnel between the timing block and the noise
parameters.

In [ ]:
noisedict = {f"{pulsar.name}_efac": 1.0}

likelihood = ds.PulsarLikelihood([
    pulsar.residuals,
    ds.makenoise_measurement_simple(pulsar, noisedict, add_equad=False),
    *timing.discovery_signals(),
])

model = nlts.numpyro.decentered_model(likelihood, timing, fixed=noisedict)
print("xi site:", model.xi_site)

## Sample

Initialising at `xi = 0` starts the chain at the conditional mode, which is the
engine's expansion point. Short chain for a notebook — scale it for science.

In [ ]:
mcmc = nlts.numpyro.nuts(
    model, timing,
    num_warmup=500, num_samples=2000, num_chains=1,
    init_strategy=init_to_value(
        values=nlts.numpyro.decentered_init_values(timing, model.transport)
    ),
)
mcmc.run(jax.random.PRNGKey(0))

print("divergences:", int(np.sum(mcmc.get_extra_fields()["diverging"])))

## The physical posterior

`posterior()` decodes the sampled coordinates back into the parameters a par
file quotes — `A1` in light-seconds, `TASC` in MJD.

In [ ]:
post = nlts.numpyro.posterior(mcmc, timing)

names = list(timing.sampled)
draws = np.column_stack(
    [np.asarray(post.posterior[n]).reshape(-1) for n in names]
)
reference = timing.space.to_physical(np.zeros(len(names)), units="display")
par_values = [float(np.asarray(reference[n]).reshape(-1)[0]) for n in names]

print(f"{'parameter':<8}{'posterior mean':>24}{'sd':>12}{'par':>24}")
for name, column, par in zip(names, draws.T, par_values):
    print(f"{name:<8}{column.mean():>24.12g}{column.std():>12.3g}{par:>24.12g}")

## Corner plot

Offsets from the par values, so six panels of very different magnitude share a
readable scale. The blue lines are the published par, which is a **reference,
not a truth**: it was fitted with a fuller noise model than the fixed
`EFAC = 1` used here, so a parameter sitting a couple of sigma off is the noise
model talking, not the delay engine.

In [ ]:
offsets = draws - np.asarray(par_values)
labels = [rf"$\Delta$ {n}" for n in names]

figure = corner.corner(
    offsets,
    labels=labels,
    truths=np.zeros(len(names)),
    truth_color="#4c72b0",
    color="#c44e52",
    levels=(0.68, 0.95),
    plot_datapoints=False,
    fill_contours=True,
    hist_kwargs={"color": "#c44e52"},
    label_kwargs={"fontsize": 12},
)
figure.suptitle(
    f"{pulsar.name} (InPTA NB, {len(pulsar.toas)} TOAs)  --  "
    "orbital parameters through vela-jax\n"
    "Discovery likelihood, decentered NUTS; offsets from the par",
    fontsize=13, y=1.02,
)
plt.show()

## Where this goes next

* **Several datasets, one timing model.** `create_metapulsar(..., engines="vela_jax")`
  builds a vela-jax leg per PTA and shares the timing parameters across them.
  See `examples/nuts_j1853.py` for EPTA (tempo2) plus NANOGrav (PINT) combined.
* **A real noise model.** Free EFAC/EQUAD/ECORR and red-noise GPs are ordinary
  Discovery signals; the timing block does not change.
* **fp32.** The perturbative engine evaluates the delta chain in float32 against
  a float64 reference, for GPUs where that matters. See `examples/nuts_fp32.py`.
* **Parity.** `examples/parity_report.py` compares vela-jax residuals against
  Vela.jl, and across the two timing packages, on a dataset of your choice.